In [2]:
import joblib
import pandas as pd

# 1. Đọc dữ liệu thô
# Vì notebook eda.ipynb nằm trong thư mục src/, nên đường dẫn cần lùi 1 thư mục bằng '../'
data = pd.read_csv('../raw_data/application_train.csv')
x = data.drop('TARGET', axis=1).copy()
y = data['TARGET'].copy()

# 2. Tải pipeline đã huấn luyện sẵn từ tệp pkl
pipeline = joblib.load('../models/baseline.pkl')

# 3. Cắt lát pipeline để chỉ lấy phần Tiền xử lý (loại bỏ mô hình LightGBM ở cuối)
preprocessor = pipeline[:-1]

# 4. Biến đổi dữ liệu thô sang dữ liệu sạch qua pipeline
print("Running preprocessing pipeline...")
x_prepped = preprocessor.transform(x)

# 5. Thêm lại cột TARGET vào tập dữ liệu sạch để tiện vẽ biểu đồ/tính tương quan
x_prepped['TARGET'] = y

print(f"Done! Clean dataset shape: {x_prepped.shape}")

Running preprocessing pipeline...
Feature encoding completed: data shape: (307511, 139)
Feature dropping completed, data shape; (307511, 102), feature dropped: 37
New features created: ['credit_income_ratio', 'income_annuity_ratio', 'birth_employed_ratio', 'credit_goods_ratio', 'EXT_mean']
Feature imputing completed.
Data scaled completed
Done! Clean dataset shape: (307511, 108)


In [ ]:
important_feature = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'AMT_ANNUITY', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE']

In [3]:
# Giả sử x_prepped là dữ liệu sạch bạn vừa tạo ra ở bước trước
# 1. Chia điểm ext_sources_mean thành 10 khoảng (deciles) bằng nhau
x_prepped['ext_mean_bin'] = pd.qcut(x_prepped['EXT_mean'], q=10, duplicates='drop')

# 2. Thống kê số lượng khách hàng và tỷ lệ nợ xấu trên từng khoảng
bin_stats = x_prepped.groupby('ext_mean_bin', observed=False)['TARGET'].agg(['count', 'mean'])
bin_stats['default_rate_%'] = bin_stats['mean'] * 100
bin_stats = bin_stats.sort_values(by='default_rate_%', ascending=False)

print("=== Thống kê tỷ lệ nợ xấu theo từng phân khúc điểm ===")
print(bin_stats)

=== Thống kê tỷ lệ nợ xấu theo từng phân khúc điểm ===
                      count      mean  default_rate_%
ext_mean_bin                                         
(-0.00099406, 0.304]  30752  0.229383       22.938345
(0.304, 0.384]        30751  0.144906       14.490586
(0.384, 0.439]        30751  0.105037       10.503723
(0.439, 0.484]        30751  0.081071        8.107053
(0.484, 0.525]        30751  0.064421        6.442067
(0.525, 0.564]        30751  0.053884        5.388443
(0.564, 0.602]        30751  0.041657        4.165718
(0.602, 0.644]        30751  0.037267        3.726708
(0.644, 0.692]        30751  0.029267        2.926734
(0.692, 0.879]        30751  0.020390        2.038958


In [ ]:
# Giả sử x_prepped là dữ liệu sạch bạn vừa tạo ra ở bước trước
# 1. Chia điểm ext_sources_mean thành 10 khoảng (deciles) bằng nhau
x_prepped['own_car_age_bin'] = pd.qcut(x_prepped['OWN_CAR_AGE'], q=5, duplicates='drop')

# 2. Thống kê số lượng khách hàng và tỷ lệ nợ xấu trên từng khoảng
bin_stats = x_prepped.groupby('own_car_age_bin', observed=False)['TARGET'].agg(['count', 'mean'])
bin_stats['default_rate_%'] = bin_stats['mean'] * 100
bin_stats = bin_stats.sort_values(by='default_rate_%', ascending=False)

print("=== Thống kê tỷ lệ nợ xấu theo từng phân khúc độ tuổi mua được xe riêng ===")
print(bin_stats)

=== Thống kê tỷ lệ nợ xấu theo từng phân khúc độ tuổi mua được xe riêng ===
                  count      mean  default_rate_%
own_car_age_bin                                  
(0.0, 82.0]       51081  0.086608        8.660754
(-9.001, 0.0]    256430  0.079558        7.955777


In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Tạo một bản sao từ biến data đã được load ở Cell 1 để phân tích
data = x_prepped.copy()
df_car = data[['FLAG_OWN_CAR', 'OWN_CAR_AGE', 'TARGET']].copy()

# 2. Phân tích 1: Tỷ lệ nợ xấu giữa nhóm Có xe (Y) vs Không có xe (N)
print("=== Tỷ lệ nợ xấu: Có xe (Y) vs Không có xe (N) ===")
car_owner_stats = df_car.groupby('FLAG_OWN_CAR')['TARGET'].agg(['count', 'mean'])
car_owner_stats['default_rate_%'] = car_owner_stats['mean'] * 100
print(car_owner_stats)

# 3. Phân tích 2: Phân nhóm tuổi thọ của xe (Bucketing)
def classify_car_age(age):
    if pd.isna(age):
        return '0. No Car'
    elif age <= 3:
        return '1. New Car (0-3 yrs)'
    elif age <= 8:
        return '2. Medium Car (3-8 yrs)'
    elif age <= 15:
        return '3. Old Car (8-15 yrs)'
    else:
        return '4. Very Old Car (>15 yrs)'

df_car['car_age_group'] = df_car['OWN_CAR_AGE'].apply(classify_car_age)

print("\n=== Tỷ lệ nợ xấu theo từng phân khúc tuổi xe ===")
car_age_stats = df_car.groupby('car_age_group')['TARGET'].agg(['count', 'mean'])
car_age_stats['default_rate_%'] = car_age_stats['mean'] * 100
car_age_stats = car_age_stats.sort_index()
print(car_age_stats)

# 4. Vẽ biểu đồ trực quan hóa kết quả
plt.figure(figsize=(10, 6))
sns.barplot(
    x=car_age_stats['default_rate_%'], 
    y=car_age_stats.index, 
    palette='Reds_d'
)
plt.title('Tỷ lệ nợ xấu (%) theo Phân khúc Tuổi Xe')
plt.xlabel('Tỷ lệ nợ xấu (%)')
plt.ylabel('Phân khúc Tuổi Xe')
plt.axvline(x=8.07, color='blue', linestyle='--', label='Tỷ lệ nợ xấu TB hệ thống (8.07%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

KeyError: "['FLAG_OWN_CAR'] not in index"